# scRNA-seq + TCR – 06 Dual TCRs in human CD4 T cells (public reference)

Processed CD4 T-cell data of the human reference by Terekhova et al. (Synapse), CITE-seq + scRNA-seq + 10x TCR:

* `cd4_rna.rds` – already normalised gene-expression matrix
* `cd4_hto.rds` – already normalised surface-protein (CITE-seq) matrix
* `cd4_metadata.csv`, `cd4_umap.csv` – author metadata / QC and UMAP
* `*_TCR_outs_filtered_contig_annotations.csv` – 10x TCR contigs

The matrices are **not** normalised again (inserted into the `data` layer). Phenotypically naive cells are gated on
CITE-seq (CD20⁻ CD3⁺ CD4⁺ CD45RA⁺ CD45RO⁻ CD62L⁺) and re-clustered (20 PCs, resolution 0.4). Dual TCR = more than
one distinct productive TRA **or** TRB chain (V gene | J gene | CDR3 nt), counted directly from the contig tables;
frequencies use only cells with a recovered TCR from samples with a matched TCR library.

**Figure panels produced here** (panel as file prefix)

| panel | content | output |
|---|---|---|
| Fig. 6g | lineage-marker dot plot per Seurat cluster | `Fig6g_DotPlot_lineage_markers_naive_clusters.pdf` |
| Fig. 6h | UMAP of the gated cells, annotated clusters | `Fig6h_UMAP_annotated_clusters.pdf` |
| Fig. 6i | dual-TCR frequency per donor and cluster | `Fig6i_dualTCR_frequency_per_donor_boxplot.pdf` |
| Suppl. Fig. 5b | UMAP coloured by Seurat cluster | `SuppFig5b_UMAP_naive_clusters.pdf` |

Statistics in Fig. 6i: each effector cluster (5, 6, 10) vs the Th0 clusters (0–3, 8), paired Wilcoxon test over donors
(≥ 20 TCR⁺ cells per donor and cluster), BH-adjusted.

## 1. Setup

In [ ]:
# ============================================================
# 0. Packages
# ============================================================

library(Seurat)
library(SeuratObject)
library(scRepertoire)
library(dplyr)
library(ggplot2)
library(stringr)
library(tidyr)
library(patchwork)
library(emmeans)

packageVersion("Seurat")
packageVersion("SeuratObject")
packageVersion("scRepertoire")

In [ ]:
# ============================================================
# 1. Paths
# ============================================================

base_dir <- "../data/Terekhova_human_CD4"   # Synapse download (processed CD4 data + TCR contigs)

data_dir <- file.path(
    base_dir,
    "cd4_t_cells_extracted",
    "cd4_t_cells"
)

tcr_dir <- base_dir

rna_file  <- file.path(data_dir, "cd4_rna.rds")
hto_file  <- file.path(data_dir, "cd4_hto.rds")
meta_file <- file.path(data_dir, "cd4_metadata.csv")
umap_file <- file.path(data_dir, "cd4_umap.csv")

stopifnot(
    file.exists(rna_file),
    file.exists(hto_file),
    file.exists(meta_file),
    file.exists(umap_file),
    dir.exists(tcr_dir)
)

fig_dir <- "../results/figures/"
dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)

list.files(data_dir)

## 2. Load the author-processed data (already normalised)

In [ ]:
# ============================================================
# 2. Load author-provided processed data
# ============================================================

rna <- readRDS(rna_file)
hto <- readRDS(hto_file)

meta <- read.csv(
    meta_file,
    row.names = 1,
    check.names = FALSE
)

umap_author <- read.csv(
    umap_file,
    row.names = 1,
    check.names = FALSE
)

cat("RNA class:", class(rna), "\n")
cat("RNA dimensions:", dim(rna), "\n\n")

cat("Surface-protein/HTO class:", class(hto), "\n")
cat("Surface-protein/HTO dimensions:", dim(hto), "\n\n")

cat("Metadata dimensions:", dim(meta), "\n")
cat("UMAP dimensions:", dim(umap_author), "\n")

head(colnames(meta), 30)
head(rownames(rna))
head(colnames(rna))
head(rownames(hto))
head(colnames(hto))

In [ ]:
# ============================================================
# 3. Orient matrices so that cells are columns
# ============================================================

orient_features_by_cells <- function(x, cell_ids, object_name = "matrix") {

    col_overlap <- sum(colnames(x) %in% cell_ids)
    row_overlap <- sum(rownames(x) %in% cell_ids)

    message(
        object_name, ": ",
        col_overlap, " metadata cell IDs in columns; ",
        row_overlap, " metadata cell IDs in rows."
    )

    if (row_overlap > col_overlap) {
        message("Transposing ", object_name, " so cells are columns.")
        x <- t(x)
    }

    x
}

rna <- orient_features_by_cells(
    rna,
    rownames(meta),
    "RNA"
)

hto <- orient_features_by_cells(
    hto,
    rownames(meta),
    "surface-protein/HTO"
)

cat("RNA cells matching metadata:",
    sum(colnames(rna) %in% rownames(meta)), "/", ncol(rna), "\n")

cat("HTO/protein cells matching metadata:",
    sum(colnames(hto) %in% rownames(meta)), "/", ncol(hto), "\n")

In [ ]:
# ============================================================
# 4. Check that the normalized RNA matrix is not raw counts
# ============================================================

# The authors describe cd4_rna.rds as a normalized matrix.
# This diagnostic is only a sanity check.

if (inherits(rna, "sparseMatrix")) {
    cat("Summary of non-zero RNA values:\n")
    print(summary(rna@x))

    cat(
        "All non-zero values are integers:",
        all(rna@x == round(rna@x)),
        "\n"
    )
}

## 3. Seurat object from the normalised matrices

RNA → `data` layer of the RNA assay; surface protein → `data` layer of an assay called `HTO` (naming of the source file).
Author metadata / QC are used as provided; the author UMAP is attached for the global overview.

In [ ]:
# ============================================================
# 5. Build Seurat object from NORMALIZED RNA data
# ============================================================

# Keep the RNA cells for which author metadata exist.
rna_cells <- intersect(
    colnames(rna),
    rownames(meta)
)

rna <- rna[, rna_cells, drop = FALSE]
meta_use <- meta[rna_cells, , drop = FALSE]

# Create an Assay5 containing a normalized data layer only.
rna_assay <- CreateAssay5Object(
    data = rna
)

seu <- CreateSeuratObject(
    counts = rna_assay,
    meta.data = meta_use,
    project = "Terekhova_CD4"
)

DefaultAssay(seu) <- "RNA"

seu
Layers(seu[["RNA"]])

In [ ]:
# ============================================================
# 6. Add normalized surface-protein / HTO assay
# ============================================================

hto_overlap <- intersect(
    colnames(seu),
    colnames(hto)
)

cat(
    "RNA cells with matching HTO/protein data:",
    length(hto_overlap), "/", ncol(seu),
    "\n"
)

if (length(hto_overlap) != ncol(seu)) {
    warning(
        "Not every RNA cell has a matching surface-protein barcode. ",
        "The HTO assay will only be added if all cells match."
    )
} else {

    hto <- hto[, colnames(seu), drop = FALSE]

    seu[["HTO"]] <- CreateAssay5Object(
        data = hto
    )
}

Assays(seu)

if ("HTO" %in% Assays(seu)) {
    Layers(seu[["HTO"]])
    head(rownames(seu[["HTO"]]))
}

In [ ]:
# ============================================================
# 8. Attach author-provided UMAP
# ============================================================

# Orient UMAP so cells are rows if needed.
row_overlap <- sum(rownames(umap_author) %in% colnames(seu))
col_overlap <- sum(colnames(umap_author) %in% colnames(seu))

if (col_overlap > row_overlap) {
    umap_author <- as.data.frame(t(as.matrix(umap_author)))
}

cat(
    "Author UMAP cells matching Seurat:",
    sum(rownames(umap_author) %in% colnames(seu)),
    "/", ncol(seu),
    "\n"
)

missing_umap <- setdiff(
    colnames(seu),
    rownames(umap_author)
)

if (length(missing_umap) > 0) {
    stop(
        "Some Seurat cells are missing from cd4_umap.csv: ",
        length(missing_umap)
    )
}

umap_author <- umap_author[
    colnames(seu),
    ,
    drop = FALSE
]

colnames(umap_author)[1:2] <- c(
    "UMAP_1",
    "UMAP_2"
)

seu[["umap"]] <- CreateDimReducObject(
    embeddings = as.matrix(
        umap_author[, c("UMAP_1", "UMAP_2")]
    ),
    key = "UMAP_",
    assay = "RNA"
)
options(repr.plot.width=6, repr.plot.height=4)
DimPlot(
    seu,
    reduction = "umap",
    group.by = "Cluster_names",
    raster = TRUE
)

### Tfh vs Th1 markers of the author annotation (supporting table, not in the manuscript)

Positive `avg_log2FC` = higher in Tfh.

In [ ]:
DefaultAssay(seu) <- "RNA"

# Check that the required author annotations are present
table(seu$Cluster_names)

stopifnot(
    all(c("Tfh", "Th1") %in% unique(seu$Cluster_names))
)

# Restrict to the two populations of interest
tfh_th1 <- subset(
    seu,
    subset = Cluster_names %in% c("Tfh", "Th1")
)

Idents(tfh_th1) <- "Cluster_names"

table(Idents(tfh_th1))

In [ ]:
# ============================================================
# Differential expression: Tfh vs Th1
#
# Positive avg_log2FC = higher in Tfh
# Negative avg_log2FC = higher in Th1
# ============================================================

tfh_vs_th1_deg <- FindMarkers(
    tfh_th1,
    ident.1 = "Tfh",
    ident.2 = "Th1",
    assay = "RNA",
    layer = "data",
    test.use = "wilcox",
    min.pct = 0.10,
    logfc.threshold = 0.10,
    only.pos = FALSE
)

tfh_vs_th1_deg$gene <- rownames(tfh_vs_th1_deg)

tfh_vs_th1_deg <- tfh_vs_th1_deg |>
    select(
        gene,
        everything()
    ) |>
    arrange(
        desc(avg_log2FC)
    )

head(tfh_vs_th1_deg, 20)
tail(tfh_vs_th1_deg, 20)

In [ ]:
# ============================================================
# Label DEG direction
# ============================================================

tfh_vs_th1_deg <- tfh_vs_th1_deg |>
    mutate(
        direction = case_when(
            p_val_adj < 0.05 & avg_log2FC >  0.25 ~ "Tfh_up",
            p_val_adj < 0.05 & avg_log2FC < -0.25 ~ "Th1_up",
            TRUE                                  ~ "not_selected"
        )
    )

table(tfh_vs_th1_deg$direction)

In [ ]:
# ============================================================
# Save complete Tfh-vs-Th1 DEG table
# ============================================================

out_dir <- file.path(
    base_dir,
    "processed_human_CD4_TCR"
)

dir.create(
    out_dir,
    recursive = TRUE,
    showWarnings = FALSE
)

write.csv(
    tfh_vs_th1_deg,
    file.path(
        out_dir,
        "Terekhova_Tfh_vs_Th1_DEGs.csv"
    ),
    row.names = FALSE
)

## 4. TCR preprocessing

TCR file names contain an additional numeric library ID; the sample key `ALAW-AS###-#` used in the RNA cell names is
extracted from them. Several TCR files that resolve to the same RNA sample are merged. Multi-chain cells are kept.

In [ ]:
# ============================================================
# 10. Locate TCR files
# ============================================================

# Search recursively: Synapse downloads / extracted archives may place
# TCR files in nested directories.
tcr_files <- list.files(
    tcr_dir,
    pattern = "filtered_contig_annotations\\.csv$",
    full.names = TRUE,
    recursive = TRUE
)

tcr_files <- sort(unique(tcr_files))

cat("Number of TCR contig files found:", length(tcr_files), "\n")
head(basename(tcr_files), 20)

if (length(tcr_files) == 0) {
    stop("No filtered_contig_annotations.csv files found recursively in tcr_dir.")
}

# Optional: see where the files were found.
tcr_file_locations <- data.frame(
    file = basename(tcr_files),
    directory = dirname(tcr_files),
    stringsAsFactors = FALSE
)

head(tcr_file_locations, 20)

In [ ]:
# ============================================================
# 11. Harmonize TCR sample IDs to RNA naming
# ============================================================

# RNA cell IDs look like:
#   ALAW-AS044-1_AAACCTGAGAAGCCCA-1
# so the sample is everything before the final "_<10x barcode>-<gem group>".
extract_rna_sample_id <- function(cell_ids) {
    sub("_[ACGTN]+-[0-9]+$", "", cell_ids)
}

# TCR filenames occur in more than one form, e.g.
#   ALAW-79665-AS051-1-TCR_outs_filtered_contig_annotations.csv
#   ALAW-80301-AS060-1-2-TCR_outs_filtered_contig_annotations.csv
#   ALAW-AS044-1-TCR_outs_filtered_contig_annotations.csv
#
# The RNA naming uses the stable ALAW-AS###-# key.
# We therefore extract that key rather than relying on one exact filename format.
canonicalize_tcr_sample_id <- function(x) {

    x <- basename(x)

    x <- sub(
        "-TCR_outs_filtered_contig_annotations\\.csv$",
        "",
        x
    )

    m <- stringr::str_match(
        x,
        "(AS[0-9]+-[0-9]+)"
    )

    canonical <- ifelse(
        !is.na(m[, 2]),
        paste0("ALAW-", m[, 2]),
        x
    )

    canonical
}

# Raw library/file IDs.
tcr_sample_ids_raw <- sub(
    "-TCR_outs_filtered_contig_annotations\\.csv$",
    "",
    basename(tcr_files)
)

# Canonical IDs used to match RNA.
tcr_sample_ids <- canonicalize_tcr_sample_id(tcr_files)

# RNA sample IDs derived directly from the actual Seurat cell names.
rna_cell_sample <- extract_rna_sample_id(colnames(seu))
rna_sample_ids <- sort(unique(rna_cell_sample))

# Count RNA cells per sample.
rna_sample_counts <- as.data.frame(table(rna_cell_sample))
colnames(rna_sample_counts) <- c("rna_sample_id", "n_RNA_cells")
rna_sample_counts$rna_sample_id <- as.character(
    rna_sample_counts$rna_sample_id
)

# File-level mapping table.
sample_name_check <- data.frame(
    tcr_file = basename(tcr_files),
    tcr_raw = tcr_sample_ids_raw,
    tcr_corrected = tcr_sample_ids,
    stringsAsFactors = FALSE
) |>
    left_join(
        rna_sample_counts,
        by = c("tcr_corrected" = "rna_sample_id")
    ) |>
    mutate(
        present_in_rna = !is.na(n_RNA_cells)
    )

sample_name_check

cat("\nTCR file -> RNA sample matching:\n")
print(table(sample_name_check$present_in_rna))

cat("\nTCR libraries/files that do not map to a CD4 RNA sample:\n")
print(
    unique(
        sample_name_check$tcr_corrected[
            !sample_name_check$present_in_rna
        ]
    )
)

# RNA-centric check: this is the important reverse comparison.
tcr_samples_available <- sort(unique(tcr_sample_ids))

rna_sample_check <- rna_sample_counts |>
    mutate(
        TCR_file_available = rna_sample_id %in% tcr_samples_available
    ) |>
    arrange(TCR_file_available, desc(n_RNA_cells))

cat("\nRNA samples with / without a matched local TCR file:\n")
print(table(rna_sample_check$TCR_file_available))

cat("\nRNA samples without a matched local TCR file:\n")
print(
    rna_sample_check |>
        filter(!TCR_file_available)
)

# Detect cases where several TCR files collapse to the same canonical RNA sample.
tcr_files_per_sample <- sample_name_check |>
    count(
        tcr_corrected,
        name = "n_TCR_files"
    ) |>
    arrange(desc(n_TCR_files), tcr_corrected)

cat("\nCanonical samples represented by >1 TCR file:\n")
print(
    tcr_files_per_sample |>
        filter(n_TCR_files > 1)
)

In [ ]:
# ============================================================
# Bidirectional RNA <-> TCR sample matching summary
# ============================================================

# ------------------------------------------------------------
# 1. RNA -> TCR
# ------------------------------------------------------------

rna_to_tcr_summary <- rna_sample_counts %>%
    mutate(
        TCR_file_available = rna_sample_id %in% tcr_samples_available
    )

cat("\n============================\n")
cat("RNA -> TCR matching\n")
cat("============================\n")

cat(
    "RNA samples total:",
    nrow(rna_to_tcr_summary),
    "\n"
)

cat(
    "RNA samples WITH matched TCR file:",
    sum(rna_to_tcr_summary$TCR_file_available),
    "\n"
)

cat(
    "RNA samples WITHOUT matched TCR file:",
    sum(!rna_to_tcr_summary$TCR_file_available),
    "\n"
)

cat(
    "Percent RNA samples with TCR:",
    round(
        100 * mean(rna_to_tcr_summary$TCR_file_available),
        1
    ),
    "%\n"
)

# Also quantify cells, not just samples
cat(
    "\nRNA cells in samples WITH matched TCR file:",
    sum(
        rna_to_tcr_summary$n_RNA_cells[
            rna_to_tcr_summary$TCR_file_available
        ]
    ),
    "\n"
)

cat(
    "RNA cells in samples WITHOUT matched TCR file:",
    sum(
        rna_to_tcr_summary$n_RNA_cells[
            !rna_to_tcr_summary$TCR_file_available
        ]
    ),
    "\n"
)

cat(
    "Percent RNA cells from samples with TCR:",
    round(
        100 *
        sum(
            rna_to_tcr_summary$n_RNA_cells[
                rna_to_tcr_summary$TCR_file_available
            ]
        ) /
        sum(rna_to_tcr_summary$n_RNA_cells),
        1
    ),
    "%\n"
)

# ------------------------------------------------------------
# 2. TCR -> RNA
# ------------------------------------------------------------

tcr_to_rna_summary <- data.frame(
    tcr_sample_id = tcr_samples_available,
    stringsAsFactors = FALSE
) %>%
    mutate(
        RNA_sample_available = tcr_sample_id %in% rna_sample_ids
    )

cat("\n============================\n")
cat("TCR -> RNA matching\n")
cat("============================\n")

cat(
    "TCR samples total:",
    nrow(tcr_to_rna_summary),
    "\n"
)

cat(
    "TCR samples WITH matched RNA sample:",
    sum(tcr_to_rna_summary$RNA_sample_available),
    "\n"
)

cat(
    "TCR samples WITHOUT matched RNA sample:",
    sum(!tcr_to_rna_summary$RNA_sample_available),
    "\n"
)

cat(
    "Percent TCR samples with RNA:",
    round(
        100 * mean(tcr_to_rna_summary$RNA_sample_available),
        1
    ),
    "%\n"
)

# ------------------------------------------------------------
# 3. Show unmatched samples in both directions
# ------------------------------------------------------------

cat("\nRNA samples WITHOUT TCR:\n")
print(
    rna_to_tcr_summary %>%
        filter(!TCR_file_available) %>%
        arrange(desc(n_RNA_cells))
)

cat("\nTCR samples WITHOUT RNA:\n")
print(
    tcr_to_rna_summary %>%
        filter(!RNA_sample_available)
)

In [ ]:
# ============================================================
# 12. Read TCR contig files and merge by canonical RNA sample
# ============================================================

# Read each physical TCR file first.
contig_by_file <- lapply(
    seq_along(tcr_files),
    function(i) {

        x <- read.csv(
            tcr_files[i],
            stringsAsFactors = FALSE,
            check.names = FALSE
        )

        x$source_tcr_file <- basename(tcr_files[i])
        x$source_tcr_id <- tcr_sample_ids_raw[i]
        x$sample_id <- tcr_sample_ids[i]

        x
    }
)

# Merge files that resolve to the same canonical RNA sample.
# This avoids duplicate sample names being silently handled inconsistently.
contig_all <- bind_rows(contig_by_file)

contig_list <- split(
    contig_all,
    contig_all$sample_id
)

contig_list <- contig_list[
    order(names(contig_list))
]

cat("Canonical TCR samples:", length(contig_list), "\n")
head(names(contig_list))

# QC: source files contributing to each canonical sample.
contig_source_qc <- contig_all |>
    distinct(sample_id, source_tcr_file) |>
    count(sample_id, name = "n_source_files") |>
    arrange(desc(n_source_files), sample_id)

print(
    contig_source_qc |>
        filter(n_source_files > 1)
)

colnames(contig_list[[1]])
head(contig_list[[1]])

In [ ]:
# ============================================================
# 13. Raw TCR QC per canonical sample
# ============================================================

tcr_qc <- bind_rows(
    lapply(
        seq_along(contig_list),
        function(i) {

            x <- contig_list[[i]]

            prod_flag <- if ("productive" %in% colnames(x)) {
                tolower(as.character(x$productive)) %in%
                    c("true", "t", "1")
            } else {
                rep(NA, nrow(x))
            }

            data.frame(
                sample_id = names(contig_list)[i],
                n_source_files =
                    n_distinct(x$source_tcr_file),
                n_contigs = nrow(x),
                n_barcodes = n_distinct(x$barcode),
                n_productive =
                    sum(prod_flag, na.rm = TRUE),
                n_TRA =
                    if ("chain" %in% colnames(x)) {
                        sum(x$chain == "TRA", na.rm = TRUE)
                    } else {
                        NA_integer_
                    },
                n_TRB =
                    if ("chain" %in% colnames(x)) {
                        sum(x$chain == "TRB", na.rm = TRUE)
                    } else {
                        NA_integer_
                    }
            )
        }
    )
)

tcr_qc

In [ ]:
# ============================================================
# 14. Combine TCRs with scRepertoire
# ============================================================

# Keep multi-chain cells for the dual-TCR analysis.
# contig_list is now named by the canonical RNA sample ID.
combined_tcr <- combineTCR(
    contig_list,
    samples = names(contig_list),
    removeNA = FALSE,
    removeMulti = FALSE,
    filterMulti = FALSE
)

length(combined_tcr)
names(combined_tcr)
head(combined_tcr[[1]])
colnames(combined_tcr[[1]])

In [ ]:
# ============================================================
# 15. Exact RNA/TCR barcode overlap
# ============================================================

tcr_barcodes <- unique(
    unlist(
        lapply(
            combined_tcr,
            function(x) x$barcode
        )
    )
)

seurat_barcodes <- colnames(seu)

barcode_overlap <- intersect(
    seurat_barcodes,
    tcr_barcodes
)

match_summary <- data.frame(
    n_seurat_cells = length(seurat_barcodes),
    n_tcr_cells = length(tcr_barcodes),
    n_matched = length(barcode_overlap),
    pct_tcr_matched =
        100 * length(barcode_overlap) / length(tcr_barcodes),
    pct_seurat_with_tcr =
        100 * length(barcode_overlap) / length(seurat_barcodes)
)

match_summary

head(seurat_barcodes)
head(tcr_barcodes)

# Check that TCR barcodes use the same canonical sample prefixes as RNA.
tcr_barcode_sample <- extract_rna_sample_id(tcr_barcodes)

cat("\nTCR barcode sample prefixes absent from RNA:\n")
print(
    setdiff(
        unique(tcr_barcode_sample),
        rna_sample_ids
    )
)

In [ ]:
# ============================================================
# 16. Add scRepertoire clonotype information to Seurat
# ============================================================

# In this installed scRepertoire version the grouping column is "sample".
seu_tcr <- combineExpression(
    combined_tcr,
    seu,
    cloneCall = "strict",
    group.by = "sample",
    proportion = TRUE
)

seu_tcr

new_cols <- setdiff(
    colnames(seu_tcr@meta.data),
    colnames(seu@meta.data)
)

new_cols

### Explicit count of productive TRA / TRB chains per cell

Distinct chain = V gene | J gene | CDR3 (nt). Cells with > 1 distinct productive TRA and/or TRB chain are
candidate dual-TCR cells (operational definition).

In [ ]:
# ============================================================
# 17. Build productive contig table
# ============================================================

productive_contigs <- bind_rows(
    lapply(
        seq_along(contig_list),
        function(i) {

            x <- contig_list[[i]]
            sid <- names(contig_list)[i]

            if ("productive" %in% colnames(x)) {
                productive_flag <- tolower(
                    as.character(x$productive)
                ) %in% c("true", "t", "1")

                x <- x[
                    productive_flag,
                    ,
                    drop = FALSE
                ]
            }

            if (nrow(x) == 0) {
                message(
                    "Skipping sample with 0 productive contigs: ",
                    sid
                )
                return(NULL)
            }

            x$sample_id <- rep(
                sid,
                nrow(x)
            )

            x$cell_barcode <- paste0(
                sid,
                "_",
                x$barcode
            )

            x
        }
    )
)

dim(productive_contigs)
head(productive_contigs)

In [ ]:
# ============================================================
# 18. Count distinct productive TRA / TRB chains per cell
# ============================================================

# Prefer nucleotide CDR3 for distinguishing chains.
if ("cdr3_nt" %in% colnames(productive_contigs)) {
    productive_contigs$tcr_seq <- productive_contigs$cdr3_nt
} else {
    productive_contigs$tcr_seq <- productive_contigs$cdr3
}

# Add V/J identity when available.
v_part <- if ("v_gene" %in% colnames(productive_contigs)) {
    productive_contigs$v_gene
} else {
    ""
}

j_part <- if ("j_gene" %in% colnames(productive_contigs)) {
    productive_contigs$j_gene
} else {
    ""
}

productive_contigs$chain_id <- paste(
    v_part,
    j_part,
    productive_contigs$tcr_seq,
    sep = "|"
)

chain_counts <- productive_contigs |>
    filter(
        chain %in% c("TRA", "TRB")
    ) |>
    group_by(
        cell_barcode
    ) |>
    summarise(
        TRA_count = n_distinct(
            chain_id[chain == "TRA"],
            na.rm = TRUE
        ),
        TRB_count = n_distinct(
            chain_id[chain == "TRB"],
            na.rm = TRUE
        ),
        .groups = "drop"
    )

head(chain_counts)

table(chain_counts$TRA_count, useNA = "ifany")
table(chain_counts$TRB_count, useNA = "ifany")

In [ ]:
# ============================================================
# 19. Add chain counts to Seurat metadata
# ============================================================

chain_meta <- as.data.frame(chain_counts)

rownames(chain_meta) <- chain_meta$cell_barcode
chain_meta$cell_barcode <- NULL

chain_meta <- chain_meta[
    intersect(
        rownames(chain_meta),
        colnames(seu_tcr)
    ),
    ,
    drop = FALSE
]

seu_tcr <- AddMetaData(
    seu_tcr,
    metadata = chain_meta
)

seu_tcr$dual_TRA_candidate <- seu_tcr$TRA_count > 1
seu_tcr$dual_TRB_candidate <- seu_tcr$TRB_count > 1

seu_tcr$dual_TCR_candidate <- (
    seu_tcr$dual_TRA_candidate |
    seu_tcr$dual_TRB_candidate
)

table(
    seu_tcr$dual_TCR_candidate,
    useNA = "ifany"
)

table(
    seu_tcr$dual_TRA_candidate,
    seu_tcr$dual_TRB_candidate,
    useNA = "ifany"
)

In [ ]:
# ============================================================
# 20. Per-sample RNA/TCR matching QC
# ============================================================

# Evaluate ALL RNA samples, including samples for which no local TCR
# contig file was found. This separates "no matched library" from
# "library present, but no productive TCR recovered".
all_sample_ids <- sort(
    union(
        rna_sample_ids,
        names(contig_list)
    )
)

match_per_sample <- lapply(
    all_sample_ids,
    function(sid) {

        rna_cells <- colnames(seu_tcr)[
            extract_rna_sample_id(colnames(seu_tcr)) == sid
        ]

        tcr_cells <- chain_counts$cell_barcode[
            extract_rna_sample_id(chain_counts$cell_barcode) == sid
        ]

        matched <- intersect(
            rna_cells,
            tcr_cells
        )

        has_tcr_file <- sid %in% names(contig_list)

        data.frame(
            sample = sid,
            TCR_file_available = has_tcr_file,
            RNA_CD4 = length(rna_cells),
            productive_TCR_cells = length(tcr_cells),
            matched = length(matched),
            pct_RNA_with_productive_TCR =
                if (length(rna_cells) > 0) {
                    100 * length(matched) / length(rna_cells)
                } else {
                    NA_real_
                }
        )
    }
)

match_per_sample <- bind_rows(match_per_sample)

match_per_sample |>
    arrange(
        TCR_file_available,
        pct_RNA_with_productive_TCR
    )

cat("\nSamples with a matched local TCR file:\n")
print(table(match_per_sample$TCR_file_available))

cat("\nRecovery among RNA samples with a matched TCR file:\n")
print(
    summary(
        match_per_sample$pct_RNA_with_productive_TCR[
            match_per_sample$TCR_file_available &
            match_per_sample$RNA_CD4 > 0
        ]
    )
)

## 5. Phenotypically naive CD4 T cells (CITE-seq gate)

Gate: CD20 < 1.0, CD3 > 1.5, CD4 > 1.5, CD45RA > 0.5, CD45RO < 1.0, CD62L > 1.5 (normalised CITE-seq values).

In [ ]:
# ============================================================
# Define CITE-seq markers for phenotypically naive CD4 T cells
#
# PI definition:
# CD20- CD3+ CD4+ CD45RA+ CD45RO- CD62L+
# ============================================================

DefaultAssay(seu_tcr) <- "HTO"

cite_features <- c(
    CD20   = "CD20 Antibody",
    CD3    = "CD3 Antibody",
    CD4    = "CD4 Antibody",
    CD45RA = "CD45RA Antibody",
    CD45RO = "CD45RO Antibody",
    CD62L  = "CD62L Antibody"
)

# Check that all requested antibodies are present
stopifnot(
    all(cite_features %in% rownames(seu_tcr[["HTO"]]))
)

cite_features

In [ ]:
# ============================================================
# Extract normalized CITE-seq expression
# ============================================================

cite_expr <- FetchData(
    seu_tcr,
    vars = unname(cite_features),
    layer = "data"
)

colnames(cite_expr) <- names(cite_features)

summary(cite_expr)

In [ ]:
# ============================================================
# Plot CITE-seq marker distributions
# ============================================================

library(ggplot2)
library(tidyr)
library(dplyr)

cite_long <- cite_expr |>
    tibble::rownames_to_column("cell") |>
    pivot_longer(
        cols = -cell,
        names_to = "marker",
        values_to = "expression"
    )

options(
    repr.plot.width = 12,
    repr.plot.height = 7
)

ggplot(
    cite_long,
    aes(x = expression)
) +
    geom_histogram(
        bins = 100
    ) +
    facet_wrap(
        ~ marker,
        scales = "free",
        ncol = 3
    ) +
    theme_classic() +
    labs(
        x = "Normalized CITE-seq expression",
        y = "Number of cells"
    )

In [ ]:
# ============================================================
# CITE-seq gating plots
# ============================================================

library(ggplot2)
library(patchwork)

cite_df <- as.data.frame(cite_expr)

# Downsample only for plotting because ~900k cells is excessive
set.seed(1)

plot_df <- cite_df[
    sample(
        seq_len(nrow(cite_df)),
        min(100000, nrow(cite_df))
    ),
]

p1 <- ggplot(
    plot_df,
    aes(x = CD3, y = CD4)
) +
    geom_hex(bins = 100) +
    theme_classic() +
    labs(
        title = "CD3 vs CD4",
        x = "CD3",
        y = "CD4"
    )

p2 <- ggplot(
    plot_df,
    aes(x = CD45RA, y = CD45RO)
) +
    geom_hex(bins = 100) +
    theme_classic() +
    labs(
        title = "CD45RA vs CD45RO",
        x = "CD45RA",
        y = "CD45RO"
    )

p3 <- ggplot(
    plot_df,
    aes(x = CD45RA, y = CD62L)
) +
    geom_hex(bins = 100) +
    theme_classic() +
    labs(
        title = "CD45RA vs CD62L",
        x = "CD45RA",
        y = "CD62L"
    )

p4 <- ggplot(
    plot_df,
    aes(x = CD20, y = CD3)
) +
    geom_hex(bins = 100) +
    theme_classic() +
    labs(
        title = "CD20 vs CD3",
        x = "CD20",
        y = "CD3"
    )

(p1 | p2) / (p3 | p4)

In [ ]:
# ============================================================
# 1. Add CITE-seq values to metadata
# ============================================================

cite_df <- as.data.frame(cite_expr)

# make sure rows are in the same order as Seurat cells
cite_df <- cite_df[colnames(seu_tcr), , drop = FALSE]

seu_tcr <- AddMetaData(
    seu_tcr,
    metadata = cite_df
)

In [ ]:
# ============================================================
# 2. First-pass CITE-seq naive gate
#
# CD20- CD3+ CD4+ CD45RA+ CD45RO- CD62L+
# ============================================================

seu_tcr$CITE_naive <- with(
    seu_tcr@meta.data,

    CD20   < 1.0 &
    CD3    > 1.5 &
    CD4    > 1.5 &
    CD45RA > 0.5 &
    CD45RO < 1.0 &
    CD62L  > 1.5
)

table(seu_tcr$CITE_naive)
prop.table(table(seu_tcr$CITE_naive))

Optional: save / reload the object with TCR and CITE-seq gate (the original analysis was continued from the saved object).

In [ ]:
seu_tcr_rds <- file.path(base_dir, "seurat_object.rds")
# saveRDS(seu_tcr, file = seu_tcr_rds)
# seu_tcr <- readRDS(seu_tcr_rds)

In [ ]:
# ============================================================
# 3. Subset phenotypically naive CD4 T cells
# ============================================================

naive <- subset(
    seu_tcr,
    subset = CITE_naive == TRUE
)

DefaultAssay(naive) <- "RNA"

naive

In [ ]:
length(unique(naive@meta.data$Donor_id))

In [ ]:
table(
    naive$Cluster_names
)

round(
    100 * prop.table(
        table(naive$Cluster_names)
    ),
    1
)

## 6. Re-clustering of the gated cells (no re-normalisation)

2,000 variable genes (`dispersion`, works on the normalised layer), scaling, PCA (30 PCs), neighbours on 20 PCs,
Louvain resolution 0.4, UMAP.

In [ ]:
# ============================================================
# 22. Recluster naive cells WITHOUT re-normalizing
# ============================================================

# The input RNA matrix is already normalized.
# "dispersion" operates on the normalized expression layer and
# avoids requiring a raw-count layer for VST.

naive <- FindVariableFeatures(
    naive,
    selection.method = "dispersion",
    nfeatures = 2000,
    verbose = FALSE
)

naive <- ScaleData(
    naive,
    features = VariableFeatures(naive),
    verbose = FALSE
)

naive <- RunPCA(
    naive,
    features = VariableFeatures(naive),
    npcs = 30,
    verbose = FALSE
)

ElbowPlot(
    naive,
    ndims = 30
)

In [ ]:
# ============================================================
# 23. Naive-only neighbors, clusters and UMAP
# ============================================================

naive <- FindNeighbors(
    naive,
    dims = 1:20,
    verbose = FALSE
)

naive <- FindClusters(
    naive,
    resolution = 0.4,
    verbose = FALSE
)

naive <- RunUMAP(
    naive,
    dims = 1:20,
    verbose = FALSE
)

DimPlot(
    naive,
    reduction = "umap",
    group.by = "seurat_clusters",
    label = F,
    raster = TRUE,raster.dpi = c(3000, 3000), pt.size = 9)

DimPlot(
    naive,
    reduction = "umap",
    group.by = "Cluster_names",
    label = TRUE,
    raster = TRUE
)

**Supplementary Figure 5b** – UMAP coloured by Seurat cluster (colour gradient from naive to effector clusters).

In [ ]:
# ------------------------------------------------------------
# 1. Cluster order: light (naive) -> dark (effector)
# ------------------------------------------------------------
cl_order <- c("0", "1", "2", "3", "8",     # naive
              "7", "9", "4",               # precursor Tfh/Tph, resting Tfh-like, memory/Tph
              "6", "5", "10",              # Treg/Tfr, cTfh, Th1
              "11")                        # other
cl_order <- cl_order[cl_order %in% levels(naive$seurat_clusters)]

# ------------------------------------------------------------
# 2. Single-hue gradient (choose one)
# ------------------------------------------------------------
ramp <- colorRampPalette(c("#deebf7", "#9ecae1", "#4292c6", "#08306b"))   # blues
# ramp <- colorRampPalette(c("#f0f0f0", "#bdbdbd", "#737373", "#252525")) # greys

cl_cols <- setNames(ramp(length(cl_order)), cl_order)

# legend sorted by cluster number, colors by differentiation
naive$seurat_clusters <- factor(naive$seurat_clusters,
                                levels = as.character(sort(as.numeric(levels(naive$seurat_clusters)))))

# ------------------------------------------------------------
# 3. UMAP
# ------------------------------------------------------------
FS <- 9
umap_theme <- theme(
  plot.title = element_blank(),
  axis.title = element_text(size = FS), axis.text = element_text(size = FS - 1),
  legend.text = element_text(size = FS), legend.title = element_blank(),
  legend.key.size = unit(10, "pt")
)

options(repr.plot.width = 5, repr.plot.height = 4)
p_naive_umap <- DimPlot(
  naive,
  reduction = "umap",
  group.by = "seurat_clusters",
  cols = cl_cols,
  order = rev(cl_order),         # dark (effector) clusters drawn on top
  label = FALSE,
  raster = TRUE, raster.dpi = c(3000, 3000), pt.size = 9
) + umap_theme

p_naive_umap

ggsave(
  filename = paste0(fig_dir, "SuppFig5b_UMAP_naive_clusters.pdf"),
  plot = p_naive_umap,
  width = 4, height = 3.5, units = "in",
  device = cairo_pdf
)

### Cluster markers (supporting, not in the manuscript)

In [ ]:
# ============================================================
# 24. Top cluster markers
# ============================================================

Idents(naive) <- "seurat_clusters"

naive_markers <- FindAllMarkers(
    naive,
    only.pos = TRUE,
    min.pct = 0.10,
    logfc.threshold = 0.25
)

naive_markers_clean <- naive_markers |>
    filter(
        p_val_adj < 0.05,
        !grepl(
            "^RPL|^RPS|^MT-|^MALAT1",
            gene,
            ignore.case = TRUE
        )
    )

top10_naive_markers <- naive_markers_clean |>
    group_by(cluster) |>
    slice_max(
        order_by = avg_log2FC,
        n = 10,
        with_ties = FALSE
    ) |>
    ungroup()

top10_genes <- unique(
    top10_naive_markers$gene
)

top10_naive_markers

In [ ]:
# ============================================================
# 25. Dotplot: top 10 markers per naive cluster
# ============================================================

options(
    repr.plot.width = 30,
    repr.plot.height = 5.5
)

DotPlot(
    naive,
    features = top10_genes,
    group.by = "seurat_clusters"
) +
    RotatedAxis() +
    theme(
        axis.text.x = element_text(
            angle = 45,
            hjust = 1,
            size = 10
        ),
        axis.text.y = element_text(size = 11)
    )+

  # Color scale: low=blue, mid=white, high=red
  scale_colour_gradient2(
    low = "blue",
    mid = "white",
    high = "red",
    midpoint = 0
  ) +

  scale_size_continuous(range = c(0.5, 6)) +

  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, size = 12),
    axis.text.y = element_text(size = 12),
    plot.title = element_text(size = 16)
  )

**Figure 6g** – naive / Tfh / Th1 lineage markers per cluster.

In [ ]:
# ============================================================
# 26. Targeted naive / Tfh / Th1 marker dotplot
# ============================================================

lineage_markers <- c(
    # Naive / early
    "TCF7", "LEF1", "CCR7", "MAL", "S1PR1", "KLF2",

    # Tfh-associated
    "PDCD1", "CXCR5", "BCL6", "IL21", "MAF",
    "CXCL13", "SOX4", "RGS1", "TOX2", "ICOS",

    # Th1-associated
    "TBX21", "IFNG", "CXCR3", "S100A4","IL2RA", "FOXP3", "PRDM1"
)

lineage_markers <- lineage_markers[
    lineage_markers %in% rownames(naive)
]

options(
    repr.plot.width = 10,
    repr.plot.height = 5
)

p_lineage <- DotPlot(

    naive,

    features = lineage_markers,

    group.by = "seurat_clusters"

) +

    RotatedAxis() +

    # Color scale: low = blue, mid = white, high = red

    scale_colour_gradient2(

        low = "blue",

        mid = "white",

        high = "red",

        midpoint = 0

    ) +

    # Adjust dot size scale
    scale_size_continuous(
        range = c(0.5, 6)
    ) +
    theme(
        axis.text.x = element_text(
            angle = 45,
            hjust = 1,
            size = 12
        ),
        axis.text.y = element_text(
            size = 12
        ),
        plot.title = element_text(
            size = 16
        )
    )

p_lineage

# ============================================================

# Save as PDF

# ============================================================

ggsave(

    filename = paste0(fig_dir, "Fig6g_DotPlot_lineage_markers_naive_clusters.pdf"),
    plot = p_lineage,
    width = 7,
    height = 4,
    units = "in",
    device = cairo_pdf

)

### QC per cluster (author QC columns)

In [ ]:
VlnPlot(
    naive,
    features = c(
        "nFeature_RNA",
        "nCount_RNA",
        "percent.mt"
    ),
    group.by = "seurat_clusters",
    pt.size = 0,
    ncol = 3
)

In [ ]:
library(dplyr)

qc_by_cluster <- naive@meta.data %>%
    group_by(seurat_clusters) %>%
    summarise(
        n_cells = n(),

        median_nFeature = median(
            nFeature_RNA,
            na.rm = TRUE
        ),

        median_nCount = median(
            nCount_RNA,
            na.rm = TRUE
        ),

        median_percent_mt = median(
            percent.mt,
            na.rm = TRUE
        ),

        mean_percent_mt = mean(
            percent.mt,
            na.rm = TRUE
        ),

        pct_mt_gt_5 = mean(
            percent.mt > 5,
            na.rm = TRUE
        ) * 100,

        pct_mt_gt_10 = mean(
            percent.mt > 10,
            na.rm = TRUE
        ) * 100
    ) %>%
    arrange(seurat_clusters)

qc_by_cluster

## 7. TCR status of the gated cells

`NA` is not "no TCR": several RNA samples have no matched TCR library. Status: dual TCR / single TCR /
no recovered TCR (library present) / no matched TCR library.

In [ ]:
# ============================================================
# 27. Define TCR-covered samples
# ============================================================

# Reuse exactly the same parser as above.
naive$sample_id_clean <- extract_rna_sample_id(
    colnames(naive)
)

# A library is considered available only when a local contig file
# maps to the same canonical RNA sample ID.
matched_tcr_samples <- intersect(
    names(contig_list),
    rna_sample_ids
)

naive$TCR_library_available <- (
    naive$sample_id_clean %in% matched_tcr_samples
)

table(
    naive$TCR_library_available,
    useNA = "ifany"
)

# Report which naive RNA samples are driving the "no matched TCR library" cells.
naive_library_qc <- data.frame(
    sample_id = naive$sample_id_clean,
    TCR_library_available = naive$TCR_library_available
) |>
    count(
        sample_id,
        TCR_library_available,
        name = "n_naive_cells"
    ) |>
    arrange(
        TCR_library_available,
        desc(n_naive_cells)
    )

naive_library_qc

In [ ]:
# ============================================================
# 28. Define TCR status
# ============================================================

naive$dual_TCR <- case_when(

    !naive$TCR_library_available ~
        "No matched TCR library",

    naive$TCR_library_available &
        is.na(naive$TRA_count) &
        is.na(naive$TRB_count) ~
        "No recovered TCR",

    naive$TRA_count > 1 |
        naive$TRB_count > 1 ~
        "Dual TCR",

    TRUE ~
        "Single TCR"
)

naive$dual_TCR <- factor(
    naive$dual_TCR,
    levels = c(
        "Dual TCR",
        "Single TCR",
        "No recovered TCR",
        "No matched TCR library"
    )
)

table(
    naive$dual_TCR,
    useNA = "ifany"
)

# Recovery denominator that is actually informative:
# among cells from samples with a matched TCR library.
table(
    naive$dual_TCR[
        naive$TCR_library_available
    ],
    useNA = "ifany"
)

Dual-TCR frequency per cluster, pooled over donors (table saved below; plot not in the manuscript).

In [ ]:
# ============================================================
# 31. Dual-TCR frequency per naive cluster
#
# IMPORTANT:
# denominator = cells with a successfully recovered TCR only
# ============================================================

dual_fraction_by_cluster <- naive@meta.data |>
    filter(
        dual_TCR %in% c(
            "Dual TCR",
            "Single TCR"
        )
    ) |>
    group_by(
        seurat_clusters
    ) |>
    summarise(
        n_TCR = n(),
        n_dual = sum(dual_TCR == "Dual TCR"),
        n_single = sum(dual_TCR == "Single TCR"),
        pct_dual = 100 * n_dual / n_TCR,
        .groups = "drop"
    ) |>
    arrange(
        desc(pct_dual)
    )

dual_fraction_by_cluster
options(
    repr.plot.width = 6,
    repr.plot.height = 6
)
p_dual_cluster <- ggplot(
    dual_fraction_by_cluster,
    aes(
        x = pct_dual,
        y = reorder(
            factor(seurat_clusters),
            pct_dual
        )
    )
) +
    geom_col() +
    labs(
        x = "Candidate dual-TCR cells among TCR-matched cells (%)",
        y = "Naive Seurat cluster"
    ) +
    theme_classic()

ggsave(

    filename = paste0(fig_dir, "F_dualTCR_frequency_per_cluster_pooled.pdf"),

    plot = p_dual_cluster,

    width = 6,

    height = 6,

    units = "in",

    device = cairo_pdf

)

## 8. Cluster annotation → Figure 6h

Clusters 0–3 and 8 = Th0 (naive), 4 = Memory/Tph, 5 = cTfh, 6 = Treg/Tfr, 7 = Precursor Tfh/Tph,
9 = Resting Tfh-like, 10 = Th1, 11 = Other.

In [ ]:
cluster_annotation <- c(
  "0" = "Th0", "1" = "Th0", "2" = "Th0", "3" = "Th0",
  "4" = "Memory/Tph", "5" = "cTfh", "6" = "Treg/Tfr", "7" = "Precursor Tfh/Tph",
  "8" = "Th0", "9" = "Resting Tfh-like", "10" = "Th1", "11" = "Other"
)

naive$celltype <- unname(cluster_annotation[as.character(naive$seurat_clusters)])

celltype_levels <- c("Th0", "Precursor Tfh/Tph", "Resting Tfh-like", "Memory/Tph",
                     "cTfh", "Treg/Tfr", "Th1", "Other")
naive$celltype <- factor(naive$celltype,
                         levels = intersect(celltype_levels, unique(naive$celltype)))

celltype_cols <- c(
  "Th0"               = "#a1d99b",   # green
  "Precursor Tfh/Tph" = "#c6dbef",   # very light blue
  "Resting Tfh-like"  = "#6baed6",   # light blue
  "Memory/Tph"        = "#4292c6",   # mid blue
  "cTfh"              = "#08519c",   # dark blue
  "Treg/Tfr"          = "#756bb1",   # purple
  "Th1"               = "#a50f15",   # dark red
  "Other"             = "#bdbdbd"    # grey
)

table(naive$seurat_clusters, naive$celltype)   # check mapping

options(repr.plot.width = 6, repr.plot.height = 4.5)
p_umap_ct <- DimPlot(
  naive,
  reduction = "umap",
  group.by = "celltype",
  cols = celltype_cols,
  label = FALSE,
  raster = TRUE, raster.dpi = c(3000, 3000), pt.size = 9
) +
  ggtitle(NULL) +
  theme(legend.text = element_text(size = 9),
        axis.title = element_text(size = 9),
        axis.text = element_text(size = 8))

p_umap_ct

ggsave(
  filename = paste0(fig_dir, "Fig6h_UMAP_annotated_clusters.pdf"),
  plot = p_umap_ct,
  width = 5, height = 3.5, units = "in",
  device = cairo_pdf
)

# Variant with labels on the UMAP (no legend)
p_umap_ct_lab <- DimPlot(
  naive, reduction = "umap", group.by = "celltype", cols = celltype_cols,
  label = F, repel = TRUE, label.size = 3,
  raster = TRUE, raster.dpi = c(3000, 3000), pt.size = 9
) + ggtitle(NULL) + NoLegend()

p_umap_ct_lab
ggsave(paste0(fig_dir, "F_UMAP_annotated_clusters_nolegend.pdf"), p_umap_ct_lab,
       width = 5, height = 4.5, units = "in", device = cairo_pdf)

## 9. Dual-TCR frequency per donor → Figure 6i

Per donor and cluster (≥ 20 cells with recovered TCR). Stars: each effector cluster (5, 6, 10) vs the Th0
clusters, paired Wilcoxon over donors, BH-adjusted.

In [ ]:
library(dplyr); library(ggplot2); library(tidyr)

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
cluster_annotation <- c(
  "0" = "Th0", "1" = "Th0", "2" = "Th0", "3" = "Th0",
  "4" = "Memory/Tph", "5" = "cTfh", "6" = "Treg/Tfr", "7" = "Precursor Tfh/Tph",
  "8" = "Th0", "9" = "Resting Tfh-like", "10" = "Th1"
)
naive_clusters    <- c("0", "1", "2", "3", "8")
effector_clusters <- c("5", "6", "10")
MIN_CELLS   <- 20
SHOW_POINTS <- TRUE

celltype_cols <- c(
  "Th0" = "#a1d99b", "Precursor Tfh/Tph" = "#c6dbef", "Resting Tfh-like" = "#6baed6",
  "Memory/Tph" = "#4292c6", "cTfh" = "#08519c", "Treg/Tfr" = "#756bb1", "Th1" = "#a50f15"
)

make_label <- function(cl) paste0(cluster_annotation[cl], " (", cl, ")")

# ------------------------------------------------------------
# Data: per donor x cluster dual-TCR %
# ------------------------------------------------------------
md <- naive@meta.data
md <- md %>% filter(dual_TCR %in% c("Dual TCR", "Single TCR"))
md$dual    <- as.integer(md$dual_TCR == "Dual TCR")
md$donor   <- factor(md$Donor_id)
md$cluster <- as.character(md$seurat_clusters)
md <- md %>% filter(cluster %in% names(cluster_annotation))

per_donor <- md %>%
  group_by(donor, cluster) %>%
  summarise(n = n(), pct = 100 * mean(dual), .groups = "drop") %>%
  filter(n >= MIN_CELLS) %>%
  mutate(celltype = cluster_annotation[cluster],
         label = make_label(cluster))

lvl <- make_label(as.character(sort(as.numeric(unique(per_donor$cluster)))))
per_donor$label <- factor(per_donor$label, levels = lvl)

# ------------------------------------------------------------
# Stats: each effector cluster vs naive clusters (paired Wilcoxon over donors, BH)
# ------------------------------------------------------------
stats <- do.call(rbind, lapply(effector_clusters, function(cl) {
  d <- md %>% filter(cluster == cl | cluster %in% naive_clusters) %>%
    mutate(eff = cluster == cl) %>%
    group_by(donor) %>%
    summarise(n_eff = sum(eff), p_eff = 100 * mean(dual[eff]), p_naive = 100 * mean(dual[!eff]),
              .groups = "drop") %>%
    filter(n_eff >= MIN_CELLS)
  data.frame(cluster = cl, n_donors = nrow(d),
             median_eff = median(d$p_eff), median_naive = median(d$p_naive),
             p = wilcox.test(d$p_eff, d$p_naive, paired = TRUE)$p.value)
}))
stats$p_adj <- p.adjust(stats$p, "BH")
stats$stars <- as.character(cut(stats$p_adj, c(-Inf, 0.001, 0.01, 0.05, Inf),
                                c("***", "**", "*", "n.s.")))
print(stats)

# star position: right of the upper whisker
star_df <- per_donor %>% filter(cluster %in% effector_clusters) %>%
  group_by(cluster, label) %>%
  summarise(y = min(max(pct), quantile(pct, 0.75) + 1.5 * IQR(pct)), .groups = "drop") %>%
  left_join(stats[, c("cluster", "stars")], by = "cluster")

# donors per cluster (console)
print(per_donor %>% count(label, name = "n_donors"))

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
p_box_cl <- ggplot(per_donor, aes(x = label, y = pct, fill = celltype)) +
  geom_boxplot(outlier.shape = NA, width = 0.65, size = 0.25, fatten = 1.5) +
  { if (SHOW_POINTS) geom_jitter(width = 0.15, height = 0, size = 0.25, alpha = 0.25) } +
  geom_text(data = star_df, aes(x = label, y = y, label = stars), inherit.aes = FALSE,
            hjust = -0.3, vjust = 0.75, size = 3) +
  scale_fill_manual(values = celltype_cols) +
  scale_y_continuous(expand = expansion(mult = c(0.01, 0.1))) +
  coord_flip() +
  labs(x = NULL, y = "Dual-TCR cell frequency (%) per donor") +
  theme_classic(base_size = 9) +
  theme(legend.position = "none",
        axis.line = element_line(size = 0.3), axis.ticks = element_line(size = 0.3))

p_box_cl
ggsave(paste0(fig_dir, "Fig6i_dualTCR_frequency_per_donor_boxplot.pdf"), p_box_cl, width = 3.4, height = 3.4, device = cairo_pdf)

### Additional statistics (not shown as figure)

(a) Th0 vs effector clusters pooled (paired Wilcoxon, quasibinomial GLM with donor);
(b) every cluster vs the mean of all clusters (quasibinomial GLM with donor, BH) and paired Wilcoxon control.

In [ ]:
# ------------------------------------------------------------
# 1. Annotation and groups
# ------------------------------------------------------------
cluster_annotation <- c(
  "0" = "Naive", "1" = "Naive", "2" = "Naive", "3" = "Naive",
  "4" = "Memory/Tph", "5" = "cTfh", "6" = "Treg/Tfr", "7" = "Precursor Tfh/Tph",
  "8" = "Naive", "9" = "Resting Tfh-like", "10" = "Th1", "11" = "Other"
)

group_map <- c(
  "0" = "Naive", "1" = "Naive", "2" = "Naive", "3" = "Naive", "8" = "Naive",
  "5" = "Effector", "6" = "Effector", "10" = "Effector"
  # 4, 7, 9, 11 excluded from this comparison
)

md <- naive@meta.data
md <- md %>% filter(dual_TCR %in% c("Dual TCR", "Single TCR"))
md$dual    <- as.integer(md$dual_TCR == "Dual TCR")
md$donor   <- factor(md$Donor_id)
md$cluster <- as.character(md$seurat_clusters)
md$group   <- unname(group_map[md$cluster])
md <- md %>% filter(!is.na(group))
md$group   <- factor(md$group, levels = c("Naive", "Effector"))

# ------------------------------------------------------------
# 2. Per donor: dual-TCR % in naive vs effector
# ------------------------------------------------------------
MIN_CELLS <- 20
per_donor <- md %>%
  group_by(donor, group) %>%
  summarise(k = sum(dual), n = n(), pct = 100 * mean(dual), .groups = "drop") %>%
  filter(n >= MIN_CELLS)

paired <- per_donor %>%
  select(donor, group, pct) %>%
  pivot_wider(names_from = group, values_from = pct) %>%
  drop_na()

plot_df <- per_donor %>% filter(donor %in% paired$donor)
summ <- plot_df %>% group_by(group) %>%
  summarise(mean = mean(pct), sd = sd(pct), n_donors = n(), .groups = "drop")
print(summ)

# ------------------------------------------------------------
# 3. Tests
# ------------------------------------------------------------
# (a) paired Wilcoxon (each donor = one pair)
wt <- wilcox.test(paired$Effector, paired$Naive, paired = TRUE)
cat(sprintf("Paired Wilcoxon (n = %d donors): p = %.2e\n", nrow(paired), wt$p.value))
cat(sprintf("Effector > Naive in %d / %d donors\n", sum(paired$Effector > paired$Naive), nrow(paired)))
cat(sprintf("Median difference: %.2f percentage points\n", median(paired$Effector - paired$Naive)))

# (b) quasibinomial GLM with donor
agg <- md %>% group_by(donor, group) %>% summarise(k = sum(dual), n = n(), .groups = "drop")
m <- glm(cbind(k, n - k) ~ group + donor, data = agg, family = quasibinomial)
print(summary(m)$coefficients["groupEffector", , drop = FALSE])
cat("OR (Effector vs Naive):", round(exp(coef(m)["groupEffector"]), 2), "\n")
print(emmeans(m, ~ group, type = "response"))

# (c) each effector cluster separately vs naive
each <- do.call(rbind, lapply(c("5", "6", "10"), function(cl) {
  d <- md %>% filter(cluster == cl | group == "Naive") %>%
    mutate(eff = cluster == cl) %>%
    group_by(donor) %>%
    summarise(n_eff = sum(eff), p_eff = mean(dual[eff]), p_naive = mean(dual[!eff]),
              .groups = "drop") %>%
    filter(n_eff >= MIN_CELLS)
  data.frame(cluster = cl, annotation = cluster_annotation[cl], n_donors = nrow(d),
             mean_eff = 100 * mean(d$p_eff), sd_eff = 100 * sd(d$p_eff),
             mean_naive = 100 * mean(d$p_naive), sd_naive = 100 * sd(d$p_naive),
             p = wilcox.test(d$p_eff, d$p_naive, paired = TRUE)$p.value)
}))
each$p_adj <- p.adjust(each$p, "BH")
print(each)

# ------------------------------------------------------------
# 4. Barplot: mean ± SD across donors, paired donor lines
# ------------------------------------------------------------
p_eff <- ggplot(plot_df, aes(group, pct)) +
  geom_col(data = summ, aes(group, mean, fill = group), width = 0.6) +
  geom_line(aes(group = donor), colour = "grey60", size = 0.2, alpha = 0.5) +
  geom_point(size = 0.6, alpha = 0.5) +
  geom_errorbar(data = summ, aes(group, ymin = pmax(mean - sd, 0), ymax = mean + sd),
                inherit.aes = FALSE, width = 0.15, size = 0.4) +
  scale_fill_manual(values = c("Naive" = "#c7e9c0", "Effector" = "#08519c")) +
  labs(x = NULL, y = "Dual-TCR cell frequency (%)\n(mean ± SD across donors)",
       subtitle = sprintf("paired Wilcoxon p = %.1e\n(n = %d donors)", wt$p.value, nrow(paired))) +
  theme_classic(base_size = 9) +
  theme(legend.position = "none", plot.subtitle = element_text(size = 7),
        axis.line = element_line(size = 0.3), axis.ticks = element_line(size = 0.3))
p_eff
ggsave(paste0(fig_dir, "F_dualTCR_effector_vs_naive.pdf"), p_eff, width = 2.2, height = 3)

In [ ]:
md <- naive@meta.data
md <- md %>% filter(dual_TCR %in% c("Dual TCR", "Single TCR"))
md$dual    <- as.integer(md$dual_TCR == "Dual TCR")
md$donor   <- factor(md$Donor_id)
md$cluster <- factor(md$seurat_clusters)
md <- md %>% filter(cluster != "11") %>% droplevels()

agg <- md %>% group_by(donor, cluster) %>%
  summarise(k = sum(dual), n = n(), .groups = "drop")

# ---------------- 1) global test ----------------
m1 <- glm(cbind(k, n - k) ~ cluster + donor, data = agg, family = quasibinomial)
m0 <- glm(cbind(k, n - k) ~ donor,           data = agg, family = quasibinomial)
print(anova(m0, m1, test = "F"))
cat("dispersion:", summary(m1)$dispersion, "\n")

# ---------------- 2) every cluster vs. mean ----------------
emm <- emmeans(m1, ~ cluster, type = "response")
eff <- contrast(emm, method = "eff", adjust = "BH")
print(eff)

eff_df <- as.data.frame(eff)
eff_df$cluster <- sub(" effect", "", eff_df$contrast)
eff_df$stars <- as.character(cut(eff_df$p.value, c(-Inf, .001, .01, .05, Inf),
                                 c("***", "**", "*", "")))

# ---------------- 3) control: paired Wilcoxon ----------------
wil <- do.call(rbind, lapply(levels(md$cluster), function(cl) {
  d <- md %>% group_by(donor) %>%
    summarise(n_in = sum(cluster == cl),
              p_in = mean(dual[cluster == cl]),
              p_out = mean(dual[cluster != cl]), .groups = "drop") %>%
    filter(n_in >= 20)
  data.frame(cluster = cl, n_donors = nrow(d),
             median_diff_pct = 100 * median(d$p_in - d$p_out),
             p = wilcox.test(d$p_in, d$p_out, paired = TRUE)$p.value)
}))
wil$p_adj <- p.adjust(wil$p, "BH")
print(wil)

# ---------------- 4) Plot ----------------
agg_plot <- agg %>% filter(n >= 20) %>% mutate(pct = 100 * k / n)

sum_df <- agg_plot %>% group_by(cluster) %>%
  summarise(mean = mean(pct), sem = sd(pct) / sqrt(n()), .groups = "drop") %>%
  left_join(eff_df[, c("cluster", "stars")], by = "cluster")

options(repr.plot.width = 4.5, repr.plot.height = 4)
p <- ggplot() +
  geom_col(data = sum_df, aes(cluster, mean, fill = cluster), width = 0.8) +
  geom_point(data = agg_plot, aes(cluster, pct), position = position_jitter(0.2, 0),
             size = 0.5, alpha = 0.35) +
  geom_errorbar(data = sum_df, aes(cluster, ymin = mean - sem, ymax = mean + sem), width = 0.3) +
  geom_text(data = sum_df, aes(cluster, mean + sem + 1.5, label = stars), hjust = 0, size = 4) +
  coord_flip() + theme_classic() + guides(fill = "none") +
  labs(x = "Seurat clusters", y = "Dual-TCR cell frequencies (%)")
p
ggsave(paste0(fig_dir, "F_dualTCR_per_cluster_donor_stats.pdf"), p, width = 4.5, height = 4)

## 10. Save objects and QC tables

In [ ]:
# ============================================================
# 32. Save processed objects and QC tables
# ============================================================

out_dir <- file.path(
    base_dir,
    "processed_human_CD4_TCR"
)

dir.create(
    out_dir,
    recursive = TRUE,
    showWarnings = FALSE
)

saveRDS(
    seu_tcr,
    file.path(out_dir, "Terekhova_CD4_author_processed_with_TCR.rds")
)

saveRDS(
    naive,
    file.path(out_dir, "Terekhova_naive_CD4_reclustered_with_TCR.rds")
)

write.csv(
    sample_name_check,
    file.path(out_dir, "TCR_sample_name_matching.csv"),
    row.names = FALSE
)

write.csv(
    match_per_sample,
    file.path(out_dir, "RNA_TCR_match_per_sample.csv"),
    row.names = FALSE
)

write.csv(
    dual_fraction_by_cluster,
    file.path(out_dir, "dual_TCR_frequency_by_naive_cluster.csv"),
    row.names = FALSE
)

write.csv(
    rna_sample_check,
    file.path(out_dir, "RNA_samples_TCR_file_availability.csv"),
    row.names = FALSE
)

write.csv(
    tcr_files_per_sample,
    file.path(out_dir, "TCR_files_per_canonical_sample.csv"),
    row.names = FALSE
)

write.csv(
    naive_library_qc,
    file.path(out_dir, "naive_TCR_library_availability_by_sample.csv"),
    row.names = FALSE
)

sessionInfo()